In [ ]:
import sys
sys.version

!pip install --upgrade pip
!pip install jupyterlab pdfplumber faiss-cpu sentence-transformers transformers torch

In [1]:
import pdfplumber
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Everything works")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Everything works


In [2]:
ollama pull llama3

SyntaxError: invalid syntax (2452604116.py, line 1)

In [2]:
import pdfplumber

def load_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

text = load_pdf("20251205.pdf")
print(text)


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


News
05 décembre 2025| #80
LETTER
SANTÉ MENTALE
AU PROGRAMME
Prendre soin de votre santé mentale, c’est essentiel.
Nous vous rappelons que Malakoff Humanis renforce son
Santé mentale
accompagnement pour faciliter l’accès aux consultations
XpeUps psychologiques :
Téléconsultation avec un psychologue : rendez-
Événement tech’ de
vous possible dans la journée, 7j/7 de 6h à minuit, avec
décembre
2 séances prises en charge intégralement jusqu’au
FlutterLille chez ADEO 31/12/2023.
Remboursement des consultations en présentiel :
Formation Agile
vos garanties couvrent les séances non remboursées
Fruits et légumes du mois par l’Assurance Maladie.
Dispositif “MonPsy” : jusqu’à 8 consultations chez un
XpeApp psychologue partenaire, remboursées à 100 % par
l’Assurance Maladie et Malakoff Humanis.
Cohérence cardiaque
Concours logo Ce dispositif sera également disponible via Benefiz.
XPEHO vous souhaite de très belles fêtes de fin d’année ! 🎄
Que ces moments soient remplis de joie, de partage et de


In [3]:
!pip install PyMuPDF
import fitz

def extract_images(pdf_path):
    doc = fitz.open(pdf_path)
    image_files = []
    
    for page_number, page in enumerate(doc, start=1):
        for img_index, img in enumerate(page.get_images(full=True), start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]
            file_name = f"page{page_number}_img{img_index}.{image_ext}"
            with open(file_name, "wb") as f:
                f.write(image_bytes)
            image_files.append(file_name)
    return image_files

images = extract_images("20251205.pdf")
print("20251205:", images)


20251205: ['page1_img1.jpeg', 'page1_img2.png', 'page2_img1.png', 'page2_img2.jpeg', 'page2_img3.jpeg', 'page3_img1.png', 'page3_img2.png', 'page3_img3.png', 'page4_img1.png', 'page4_img2.jpeg', 'page4_img3.png', 'page5_img1.png', 'page5_img2.jpeg', 'page5_img3.jpeg', 'page5_img4.png', 'page6_img1.png', 'page6_img2.jpeg', 'page6_img3.jpeg', 'page7_img1.png', 'page7_img2.png', 'page7_img3.png', 'page7_img4.png', 'page7_img5.jpeg', 'page7_img6.png', 'page7_img7.png', 'page7_img8.png', 'page7_img9.png', 'page7_img10.png']


In [26]:
def chunk_text(text, size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+size])
        start += size - overlap
    return chunks

chunks = chunk_text(text)
print(f"Total text chunks: {len(chunks)}")

Total text chunks: 15


In [27]:
#embeddings
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# vec = model.encode("This is a test sentence")
# print(vec.shape)

#create embeddings
embeddings = embedder.encode(chunks)

embeddings.shape

(15, 384)

In [34]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Vectors in index:", index.ntotal)

Vectors in index: 15


In [65]:
def search(query, k=5, min_score=0.3):
    q = embedder.encode([query])
    faiss.normalize_L2(q)
    D, I = index.search(q, k)

    results = []
    for score, idx in zip(D[0], I[0]):
        if score >= min_score:
            results.append(chunks[idx])
    return results

query = "What is XPEHO"
search("What is XPEHO", min_score=0.1)

['rir une expérience toujours plus fluide.\nEt si vous ne l’aviez pas encore remarqué, une nouvelle fonctionnalité a\nfait son apparition : la Boîte à idées ! Vous pouvez désormais proposer\nvos suggestions, voter pour celles des autres et contribuer directement\nà l’évolution de XpeApp.\nMerci pour votre confiance et votre implication dans l’amélioration continue\nde XpeApp !\nRSE\nCohérence cardiaque\nXPEHO a organisé un atelier de cohérence cardiaque, une méthode simple et\naccessible qui aide à gérer le',
 ' 14h\nPour t’inscrire : c’est par ici 🙂\nMicroservice Development with AI: Best Practices - Node.js\n23 décembre : 19h30 > 20h30\nPour t’inscrire : c’est par ici 🙂\nFLUTTERLILLE\nchez ADEO\nMerci à Théo et Evan d’avoir assisté au FlutterLille (Refonte de l’application Leroy\nMerlin : un code unifié et réutilisable), le 25 novembre dernier chez ADEO. Pour\nceux qui ont loupé le résumé le voici :\n“La session était animée par l’équipe en charge de l’application commerciale de\nLer

In [48]:
print(query_vec)

[]


In [36]:
print(index)
print(index.ntotal)

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x12a2e9620> >
15


In [37]:
print(len(chunks))
print(index.ntotal)

15
15


In [38]:
print(embeddings.shape)
print(index.d)


(15, 384)
384


In [39]:
q = embedder.encode(["test"])
print(q.shape)
print(q.dtype)


(1, 384)
float32


In [41]:
q = np.array(q).astype("float32")
faiss.normalize_L2(q)

In [42]:
D, I = index.search(q, 5)
print("Scores:", D)
print("Indices:", I)

Scores: [[0.15203151 0.12156206 0.10859952 0.10357933 0.08683367]]
Indices: [[12  1 13  0 11]]


In [63]:
search("atelier", min_score=0.1)

['a dévoilé dans la newsletter de janvier… suspense ! 🎁\nProchain rendez-vous : 15 janvier\npour : la réunion d’agence\nÀ tantôt !\n',
 'ntration\net de la performance.\nCet atelier s’inscrit dans notre démarche de\nbien-être au travail, pour accompagner\nchacun à mieux gérer le stress, rester serein et\ntrouver un équilibre entre vie professionnelle\net personnelle.\nLOGO EN MODE NOEL\nPlusieurs collaborateurs ont imaginé des versions “spécial Noël” de notre logo\n🎅✨ Nous vous proposons de voter pour votre préféré ! Il vous suffit de\ntransmettre votre choix à Pauline.\nLe grand gagnant sera dévoilé dans la newsletter de janvier… suspense ',
 ' 14h\nPour t’inscrire : c’est par ici 🙂\nMicroservice Development with AI: Best Practices - Node.js\n23 décembre : 19h30 > 20h30\nPour t’inscrire : c’est par ici 🙂\nFLUTTERLILLE\nchez ADEO\nMerci à Théo et Evan d’avoir assisté au FlutterLille (Refonte de l’application Leroy\nMerlin : un code unifié et réutilisable), le 25 novembre dernier chez A